## Setup Spark Session

In [1]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab01_Enhanced") \
    .getOrCreate()


## Create DataFrame from Dictionary

In [2]:

data = [
    {"id": 1, "name": "Ali", "age": 25, "salary": 4000},
    {"id": 2, "name": "Sara", "age": 30, "salary": 7000},
    {"id": 3, "name": "Omar", "age": 22, "salary": None},
    {"id": 4, "name": "Mona", "age": None, "salary": 5000}
]

df = spark.createDataFrame(data)

df.show()
df.printSchema()


+----+---+----+------+
| age| id|name|salary|
+----+---+----+------+
|  25|  1| Ali|  4000|
|  30|  2|Sara|  7000|
|  22|  3|Omar|  null|
|null|  4|Mona|  5000|
+----+---+----+------+

root
 |-- age: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)




## 🔍 Debugging Tip
Always inspect schema using:
- printSchema()
- show()

Why?
Because wrong data types will silently break your logic later.


## Filtering Data

    show salaries greater than 4000 and age > 23

In [3]:
# Filtering records where salary > 4000 AND age > 23
filtered_df = df.filter((df["salary"] > 4000) & (df["age"] > 23))

# Display the result
filtered_df.show()


+---+---+----+------+
|age| id|name|salary|
+---+---+----+------+
| 30|  2|Sara|  7000|
+---+---+----+------+



## Renaming Columns

rename name column to be full_name

In [4]:
# Rename the column 'name' to 'full_name'
renamed_df = df.withColumnRenamed("name", "full_name")

# Display the updated DataFrame to verify
renamed_df.show()


+----+---+---------+------+
| age| id|full_name|salary|
+----+---+---------+------+
|  25|  1|      Ali|  4000|
|  30|  2|     Sara|  7000|
|  22|  3|     Omar|  null|
|null|  4|     Mona|  5000|
+----+---+---------+------+



## Reading CSV with proper schema

In [6]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define a proper explicit schema for a theoretical employee CSV file
custom_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("full_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True)
])

# Standard syntax to read a CSV with the proper enforced schema
# (We point to a dummy path as a placeholder to complete the required notebook structure)
print("Standard Spark DataFrame CSV reading syntax configuration:")
print("df_csv = spark.read.format('csv').option('header', 'true').schema(custom_schema).load('path_to_file.csv')")


Standard Spark DataFrame CSV reading syntax configuration:
df_csv = spark.read.format('csv').option('header', 'true').schema(custom_schema).load('path_to_file.csv')



## 🧪 Data Quality Check

Find:
- Rows with null salary
- Rows with null age


In [7]:
# 1. Find and display rows where salary is null
print("Rows with null salary:")
df.filter(df["salary"].isNull()).show()

# 2. Find and display rows where age is null
print("Rows with null age:")
df.filter(df["age"].isNull()).show()

Rows with null salary:
+---+---+----+------+
|age| id|name|salary|
+---+---+----+------+
| 22|  3|Omar|  null|
+---+---+----+------+

Rows with null age:
+----+---+----+------+
| age| id|name|salary|
+----+---+----+------+
|null|  4|Mona|  5000|
+----+---+----+------+



Handle null values:
- Replace null salary with 0
- Drop rows where age is null

In [8]:
# 1. Replace null values in the 'salary' column with 0
df_filled = df.fillna({"salary": 0})

# 2. Drop rows where the 'age' column contains null values
final_df = df_filled.dropna(subset=["age"])

# Display the final cleaned DataFrame
print("Final Cleaned DataFrame:")
final_df.show()

Final Cleaned DataFrame:
+---+---+----+------+
|age| id|name|salary|
+---+---+----+------+
| 25|  1| Ali|  4000|
| 30|  2|Sara|  7000|
| 22|  3|Omar|     0|
+---+---+----+------+

